## 2. Data Cleaning and Temporal harmonization

Before starting the proper Exploratory Data Analysis, data cleaning procedure to prepare raw datasets for analysis is necessary.  
This process is achievable by building a consistent hourly time series across price, load, and generation data.  
The main objective is to align heterogeneous data sources with different temporal granularities and formats into a unified analytical structure.

### 2.1. Prices dataset
For reducing granularity and make time series analysis easier, it is better to integrate period data (2021-2025) in the same table.

In [19]:
# import libraries
import pandas as pd
from pathlib import Path
import warnings

# filter User and Filter Warnings for privacy matters
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)

# define initial variables
dataset_path = Path('../dataset/raw')
YEARS = range(2023, 2027)

# create prices dataframe
prices_dfs = [
    pd.read_excel(dataset_path / f'GME-PUNi-{year}.xlsx')
    for year in YEARS
]
prices_data = pd.concat(prices_dfs, ignore_index=True)

In [20]:
prices_data

,Data,Ora,€/MWh
0,01/01/2023,1,"195,900000"
1,01/01/2023,2,"191,090000"
2,01/01/2023,3,"187,950000"
3,01/01/2023,4,"187,820000"
4,01/01/2023,5,"187,740000"
...,...,...,...
30642,2026-06-30 00:00:00,20,204.10622
30643,30/06/2026,21,198.16
30644,2026-06-30 00:00:00,22,182.0325
30645,30/06/2026,23,173.30306


In [21]:
# fast check
print(f'''
----Rows x Cols:----
{prices_data.shape}\n
----First rows:----
{prices_data.head()}\n
----Last rows:----
{prices_data.tail()}
''')


----Rows x Cols:----
(30647, 3)

----First rows:----
         Data  Ora       €/MWh
0  01/01/2023    1  195,900000
1  01/01/2023    2  191,090000
2  01/01/2023    3  187,950000
3  01/01/2023    4  187,820000
4  01/01/2023    5  187,740000

----Last rows:----
                      Data  Ora      €/MWh
30642  2026-06-30 00:00:00   20  204.10622
30643           30/06/2026   21     198.16
30644  2026-06-30 00:00:00   22   182.0325
30645           30/06/2026   23  173.30306
30646  2026-06-30 00:00:00   24   143.5925



After integrating all data from 01-01-2021 to 31-12-2025, it is necessary to check NULL values, columns data type and other inconsistencies.

In [22]:
# check what to clean
prices_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 30647 entries, 0 to 30646
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Data    30647 non-null  object
 1   Ora     30647 non-null  int64 
 2   €/MWh   30647 non-null  object
dtypes: int64(1), object(2)
memory usage: 718.4+ KB


In [23]:
print(f'''
----Date description----
{prices_data['Data'].describe()}\n
----Hour description----
{prices_data['Ora'].describe()}\n
----Prices----
{prices_data['€/MWh'].describe()}\n
''')


----Date description----
count          30647
unique          1278
top       29/10/2023
freq              25
Name: Data, dtype: object

----Hour description----
count    30647.000000
mean        12.499723
std          6.922270
min          1.000000
25%          6.500000
50%         12.000000
75%         18.000000
max         25.000000
Name: Ora, dtype: float64

----Prices----
count          30647
unique         20090
top       105,100000
freq             180
Name: €/MWh, dtype: object




From the code output, it is noticeable that in each row there are no NULL values, but also that there are other inconsistencies:
1. <code>Data</code> datatype is object instead of datetime;
2. <code>€/MWh</code> data type is object instead of float. Also, in Italy floats use colons and not dots;
3. Hours are 25 instead of 24. This is due to a correction of the hour change in October, which introduces Solar Time.


In [24]:
## Data Cleaning
# 1. Convert Data in datetime
prices_data['Data'] = pd.to_datetime(prices_data['Data'], dayfirst=True, errors='coerce')

# 2. Convert €/MWh in float
prices_data['€/MWh'] = prices_data['€/MWh'].str.replace(',', '.').astype(float)

# 3. Correct hours
prices_data['Ora'] = prices_data['Ora'].apply(lambda x: 24 if x > 24 else x) 

# create unique datetime
prices_data['datetime'] = prices_data['Data'] + pd.to_timedelta(prices_data['Ora'] - 1, unit='h')

# Rename columns and sort
prices_data = prices_data[['datetime', 'Data', '€/MWh']].rename(
    columns={
        'Data': 'Date',
        '€/MWh': 'Prices'
    }
).sort_values('datetime').reset_index(drop=True)

In [25]:
# final cleaning check
prices_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 30647 entries, 0 to 30646
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  30647 non-null  datetime64[us]
 1   Date      30647 non-null  datetime64[us]
 2   Prices    30623 non-null  float64       
dtypes: datetime64[us](2), float64(1)
memory usage: 718.4 KB


In [26]:
prices_data['datetime'].agg(['min', 'max'])

min   2023-01-01 00:00:00
max   2026-06-30 23:00:00
Name: datetime, dtype: datetime64[us]

Now that the Dtype for <code>datetime</code> and <code>Date</code> columns is right, and that the minimum value and maximum value for <code>datetime</code> is consistent, the prices data is ready to be exported in <code>/dataset/clean</code> folder.

In [27]:
from pathlib import Path

output_path = Path('../dataset/clean/updated')

# create folder if non-existent
output_path.mkdir(parents=True, exist_ok=True)

# save in CSV format
prices_data.to_csv(output_path / 'pun-index-clean.csv', index=False)